# Produit neige mensuel binaire (GLMM par saison) – coefficients R, calcul en Python

Version Python de `Model per saison1.R` + `raster binaire step1.R` (non modifiés).

**Modèle :** ton modèle publié, un modèle par saison.
- Formule :
  ```
  ThrX ~ NDSI + (NDSI|Stations) + (NDSI|AltC) + (NDSI|Landcover)
  ```
- Seuils de neige : DJF ≥ 1 cm, MAM ≥ 5 cm, JJA ≥ 15 cm, SON ≥ 15 cm.
- Les coefficients sont ajustés **une fois dans R (lme4)** par `Snow_GLMM_fit_R_coefs.R`, puis lus ici dans `snow_model/glmm_R_coefs_saisons.csv`.

**Prédiction :** comme `re.form = ~(NDSI|AltC)+(NDSI|Landcover)`, c'est-à-dire sans l'effet station :

p = 1 / (1 + exp(−[(b0 + u_AltC + u_LC) + (b1 + v_AltC + v_LC) × NDSI_max]))

**Produit final (step1), un fichier par mois** `Snow_<année>_<mois>.tif` :
- 1 = neige (p ≥ cutoff de la saison, celui qui maximise le TSS) ;
- 0 = pas de neige ;
- 2 = nuage / pas de donnée (NDSI manquant) ;
- 255 = NA (lac ou hors Suisse, d'après `Mask_all.tif`).

**Statistiques par mois** (comme `df_resultats_predi` et `df_binaire`) : probabilité min/médiane/moyenne/max et nombre et % de pixels neige / pas neige / nuage. Elles sont écrites dans un fichier Excel.

**Différences avec R :**
- Le calcul se fait directement en numpy sur toute la Suisse, en quelques secondes par mois : plus besoin de couper en deux ni de faire des stacks par saison. La saison vient du mois.
- Testé sur novembre 2020 : 98,9 % de pixels identiques à ta carte R. L'écart vient de la grille : 6200 lignes dans tes cartes R contre 6199 pour les NDSI.

In [ ]:
import os, glob, time
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Configuration section

# monthly NDSI products (band 'max' is used); several folders possible, e.g. old datacube products + STAC products
ndsi_dirs = ['../outputs/NDSI_monthly_STAC']           # add here the folder of the 1984-2021 products, e.g. '/path/to/NDSI_datacube'
ndsi_pattern = 'NDSI_%i_%02d*.tif'            # file name of a month (year, month)

coefs_csv = '../data/model/glmm_R_coefs_saisons.csv'     # from Snow_GLMM_fit_R_coefs.R
predstack_tif = '../data/sources/predstack_saison.tif'                 # band 1 = Landcover (1,2,3), band 2 = AltC (1-6)
mask_all_tif = '../data/sources/Mask_all.tif'                          # 1 = Switzerland without lakes, NA elsewhere

out_dir = '../outputs/Snow_monthly_GLMM'
aux_dir = '../data/aux'
stats_xlsx = '%s/Snow_monthly_GLMM_stats.xlsx' % out_dir
save_proba = False                            # True: also write the probability (Proba_<year>_<month>.tif)

start_year, end_year = 1984, 2026
start_month, end_month = 1, 12

# season of each month
season_of_month = {12: 'DJF', 1: 'DJF', 2: 'DJF', 3: 'MAM', 4: 'MAM', 5: 'MAM',
                   6: 'JJA', 7: 'JJA', 8: 'JJA', 9: 'SON', 10: 'SON', 11: 'SON'}

# NDSI grid (same as the NDSI products of the datacube and STAC versions)
grid_origin = (5.699957268017272, 47.85026882419969)
grid_res    = (0.0003407435000010325, 0.00034046861818428666)
grid_size   = (14234, 6199)                    # (columns, rows)
grid_transform = from_origin(grid_origin[0], grid_origin[1], grid_res[0], grid_res[1])
grid_crs = 'EPSG:4326'
W, H = grid_size

os.makedirs(out_dir, exist_ok=True)
os.makedirs(aux_dir, exist_ok=True)

## 1. Couches auxiliaires sur la grille NDSI (créées une seule fois)

`predstack_saison.tif` et `Mask_all.tif` sont sur une grille de 14234 × 6200 pixels, très proche de celle des NDSI. On les recale une fois sur la grille NDSI, au plus proche voisin puisque ce sont des classes.
- Land cover et AltC sont codés en uint8 (0 = NA).
- Le masque vaut 1 en Suisse sans les lacs, 0 ailleurs.

In [ ]:
def to_grid(src_path, band, out_path):
    """Resample one band (classes) onto the NDSI grid (nearest), uint8 with 0 = NA."""
    if not os.path.exists(out_path):
        with rasterio.open(src_path) as src:
            a = np.zeros((H, W), 'float32')
            reproject(rasterio.band(src, band), a, src_transform=src.transform, src_crs=src.crs,
                      dst_transform=grid_transform, dst_crs=grid_crs,
                      src_nodata=src.nodata, dst_nodata=0, resampling=Resampling.nearest)
        a[~np.isfinite(a) | (np.abs(a) > 100)] = 0
        prof = dict(driver='GTiff', width=W, height=H, count=1, dtype='uint8', crs=grid_crs,
                    transform=grid_transform, nodata=0, compress='deflate')
        with rasterio.open(out_path, 'w', **prof) as dst:
            dst.write(a.astype('uint8'), 1)
    with rasterio.open(out_path) as f:
        return f.read(1)

LC   = to_grid(predstack_tif, 1, '%s/landcover_ndsi_grid.tif' % aux_dir)
ALTC = to_grid(predstack_tif, 2, '%s/altc_ndsi_grid.tif' % aux_dir)
MASK = to_grid(mask_all_tif, 1, '%s/mask_all_ndsi_grid.tif' % aux_dir) == 1

print('Landcover classes', np.unique(LC), '| AltC classes', np.unique(ALTC), '| pixels in Switzerland', MASK.sum())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(20, 4))
for a, (arr, t) in zip(ax, [(LC, 'Landcover'), (ALTC, 'AltC'), (MASK, 'Mask_all')]):
    a.imshow(arr[::5, ::5], interpolation='nearest'); a.set_title(t); a.axis('off')

## 2. Coefficients du modèle (R, lme4)

Pour chaque saison, on calcule une table d'intercept et de pente pour chaque combinaison AltC × Landcover :
- intercept = b0 + u_AltC + u_LC ;
- pente = b1 + v_AltC + v_LC.

La prédiction d'un pixel est alors une simple lecture dans cette table.

In [ ]:
coefs = pd.read_csv(coefs_csv)
display(coefs[coefs.effect == 'fixed'][['season', 'snow_thr_cm', 'intercept', 'slope', 'cutoff', 'tss']])

# lookup tables [AltC, Landcover] (index 0 = NA)
models = {}
for s in ['DJF', 'MAM', 'JJA', 'SON']:
    c = coefs[coefs.season == s]
    fx = c[c.effect == 'fixed'].iloc[0]
    I = np.full((7, 4), np.nan, 'float32'); S = np.full((7, 4), np.nan, 'float32')
    alt = c[c.effect == 'AltC'].set_index('level'); lc = c[c.effect == 'Landcover'].set_index('level')
    for a in range(1, 7):
        for l in range(1, 4):
            I[a, l] = fx.intercept + alt.loc[a, 'intercept'] + lc.loc[l, 'intercept']
            S[a, l] = fx.slope + alt.loc[a, 'slope'] + lc.loc[l, 'slope']
    models[s] = dict(I=I, S=S, cutoff=float(fx.cutoff), snow_thr_cm=int(fx.snow_thr_cm))
    print('%s: snow >= %i cm, cutoff %.2f' % (s, models[s]['snow_thr_cm'], models[s]['cutoff']))

## 3. Fonctions

In [ ]:
def find_ndsi(year, month):
    """Monthly NDSI file of a month (first found in ndsi_dirs), or None."""
    for d in ndsi_dirs:
        f = sorted(glob.glob(os.path.join(d, ndsi_pattern % (year, month))))
        if f:
            return f[0]
    return None


def read_ndsi_max(path):
    """Band 'max' of a monthly NDSI product on the NDSI grid (reprojected if needed), NaN = no data."""
    with rasterio.open(path) as src:
        b = list(src.descriptions).index('max') + 1 if 'max' in src.descriptions else 1
        same = (src.width, src.height) == (W, H) and src.transform.almost_equals(grid_transform, 1e-9)
        if same:
            a = src.read(b).astype('float32')
            if src.nodata is not None:
                a[a == src.nodata] = np.nan
        else:
            a = np.full((H, W), np.nan, 'float32')
            reproject(rasterio.band(src, b), a, src_transform=src.transform, src_crs=src.crs,
                      dst_transform=grid_transform, dst_crs=grid_crs, src_nodata=src.nodata,
                      dst_nodata=np.nan, resampling=Resampling.nearest)
    a[(a > 1) | (a < -1)] = np.nan          # as in the R script
    return a


def predict_snow(ndsi, season):
    """Probability of snow (GLMM of the season) and binary product 0/1/2/255."""
    m = models[season]
    p = 1 / (1 + np.exp(-(m['I'][ALTC, LC] + m['S'][ALTC, LC] * ndsi)))   # NaN if NDSI, AltC or LC is NA
    snow = np.full((H, W), 255, 'uint8')                     # 255 = NA (lakes, outside Switzerland)
    snow[MASK] = 2                                           # 2 = cloud / no data
    ok = MASK & np.isfinite(p)
    snow[ok] = (p[ok] >= m['cutoff']).astype('uint8')       # 1 = snow, 0 = no snow
    return p, snow


def write_tif(path, arr, dtype, nodata):
    prof = dict(driver='GTiff', width=W, height=H, count=1, dtype=dtype, crs=grid_crs, transform=grid_transform,
                nodata=nodata, compress='deflate', tiled=True)
    with rasterio.open(path, 'w', **prof) as dst:
        dst.write(arr.astype(dtype), 1)


def month_stats(year, month, season, p, snow):
    """Statistics as df_resultats_predi + df_binaire of the R scripts."""
    pv = p[MASK & np.isfinite(p)]
    n_ch = int(MASK.sum())
    n1, n0, n2 = [int((snow == k).sum()) for k in (1, 0, 2)]
    return dict(Date='%i-%02d' % (year, month), Season=season,
                Min=float(pv.min()) if pv.size else np.nan, Median=float(np.median(pv)) if pv.size else np.nan,
                Mean=float(pv.mean()) if pv.size else np.nan, Max=float(pv.max()) if pv.size else np.nan,
                snow=n1, no_snow=n0, cloud=n2, NoData=int(H * W - n_ch),
                snow_pct=100 * n1 / n_ch, no_snow_pct=100 * n0 / n_ch, cloud_pct=100 * n2 / n_ch)

## 4. Test sur un mois

Le premier mois NDSI disponible. (La comparaison avec les cartes R d'origine, 98,9 % de pixels identiques en novembre 2020, est décrite dans le README.)

In [ ]:
# test on one month (the first NDSI product found)
test = None
for y in range(start_year, end_year + 1):
    for m in range(1, 13):
        if test is None and find_ndsi(y, m):
            test = (y, m)
t0 = time.time()
ndsi = read_ndsi_max(find_ndsi(*test))
p, snow = predict_snow(ndsi, season_of_month[test[1]])
print('%i-%02d done in %.0f s' % (test[0], test[1], time.time() - t0)); month_stats(test[0], test[1], season_of_month[test[1]], p, snow)

## 5. Tous les mois

- Les mois déjà calculés sont sautés.
- Un mois sans fichier NDSI est ignoré.
- Les statistiques sont ajoutées au fichier Excel à chaque mois.

In [ ]:
stats = pd.read_excel(stats_xlsx).to_dict('records') if os.path.exists(stats_xlsx) else []
done = {r['Date'] for r in stats}

for year in range(start_year, end_year + 1):
    for month in range(start_month, end_month + 1):
        out_tif = '%s/Snow_%i_%02d.tif' % (out_dir, year, month)
        if os.path.exists(out_tif):
            continue
        f = find_ndsi(year, month)
        if f is None:
            continue
        t0 = time.time()
        season = season_of_month[month]
        p, snow = predict_snow(read_ndsi_max(f), season)
        write_tif(out_tif, snow, 'uint8', 255)
        if save_proba:
            write_tif('%s/Proba_%i_%02d.tif' % (out_dir, year, month), np.where(MASK, p, np.nan), 'float32', np.nan)
        st = month_stats(year, month, season, p, snow)
        stats = [r for r in stats if r['Date'] != st['Date']] + [st]
        pd.DataFrame(stats).sort_values('Date').to_excel(stats_xlsx, index=False)
        print('%i-%02d %s: snow %.1f %% | no snow %.1f %% | cloud %.1f %% (%.0f s)'
              % (year, month, season, st['snow_pct'], st['no_snow_pct'], st['cloud_pct'], time.time() - t0))

print('ALL DONE')

In [ ]:
df_stats = pd.read_excel(stats_xlsx)
df_stats.tail(12)